# Matplotlib Chart Types

> 📘 **Python Mastery** · Module 12 — Matplotlib · Lesson 3/5

Lines, dots, bars, bins, and slices — five shapes cover almost every question a
dataset can answer. This lesson teaches you *which* shape answers *which*
question, then builds each chart on a small realistic dataset.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Match** a data question to the right chart type using the decision table
- **Create** scatter plots with per-point size (`s`), color (`c`, `cmap`) and a colorbar
- **Build** grouped, stacked, and horizontal bar charts
- **Distinguish** histograms from bar charts; use `bins=` and explain `density=True`
- **Produce** pie charts with `autopct`, `explode`, and `startangle=90` — knowing their risks

## 1. Choosing the Right Chart — the Decision Table

The chart type is not decoration; it is an *answer format*. Match the question first:

| Your question | Chart type | Function |
|---|---|---|
| How does it change over time / sequence? | Line | `plt.plot()` |
| Which category is bigger? | Bar | `plt.bar()` / `plt.barh()` |
| Do two variables move together? | Scatter | `plt.scatter()` |
| How are values distributed? | Histogram | `plt.hist()` |
| What are the parts of one whole? | Pie (with care) | `plt.pie()` |

Rule of thumb: **trend → line, comparison → bar, relation → scatter,
distribution → histogram, parts-of-whole → pie** (and pies come with caveats —
see section 7).

**Syntax:** every chart type follows the same call pattern.

```python
plt.plot(time_points, values)     # trend
plt.scatter(x_values, y_values)   # relation
plt.bar(categories, heights)      # comparison
plt.hist(raw_values)              # distribution
plt.pie(parts, labels=names)      # composition
```

**Example:** warm up with the dataset we will reuse all lesson — 12 students'
study hours vs their exam marks.

In [ ]:
# One small dataset, reused across this whole lesson
import matplotlib.pyplot as plt

study_hours = [2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 8, 9]        # hours/week
marks       = [45, 52, 55, 58, 62, 65, 70, 72, 75, 80, 86, 92]  # exam score

plt.plot(study_hours, marks, marker="o")

plt.title("Study Hours vs Exam Marks (preview)")
plt.xlabel("Hours studied per week")
plt.ylabel("Exam marks")
plt.grid(True, linestyle="--", alpha=0.5)

plt.show()

## 2. Trend → Line Chart

When the x-axis is *time* or any ordered sequence, connect the points: the eye
follows slopes instantly. Lesson 1 taught the mechanics — here it is in its
natural habitat, comparing this year's sales against last year's.

**Syntax:**
```python
plt.plot(time, series_a, label="This year")    # label= feeds the legend
plt.plot(time, series_b, linestyle="--", label="Last year")
plt.legend()
```

**Example:** bookstore sales, 2025 against 2024 — dashed for history.

In [ ]:
import matplotlib.pyplot as plt

months       = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
sales_2025   = [320, 290, 410, 380, 450, 520]
sales_2024   = [260, 310, 330, 360, 380, 400]

plt.plot(months, sales_2025, marker="o", linewidth=2.5, label="2025")
plt.plot(months, sales_2024, linestyle="--", linewidth=2, label="2024")

plt.title("Trend Question -> Line Chart")
plt.xlabel("Month")
plt.ylabel("Books sold")
plt.legend(loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)

plt.show()

## 3. Scatter Plots — Relationships Between Two Numbers

A scatter plot puts ONE dot per observation, revealing relationships, clusters,
and outliers that summary statistics hide. It is also the workhorse of ML data
exploration: features vs features, features vs target.

Each dot can encode up to four dimensions at once:

| Argument | Encodes | Type |
|---|---|---|
| `x`, `y` | position | two numeric variables |
| `s=` | dot **area** (points²) | magnitude per point |
| `c=` | color value(s) | another numeric/category variable |
| `cmap=` | which colormap translates `c` to colors | e.g. `'viridis'` |

> 🔍 **Under the Hood:** `s` is *area*, not radius — doubling `s` does NOT double
> the dot's width, it doubles how much ink it covers (width grows by ~1.4x).
> Internally `scatter()` doesn't draw point-by-point: it builds one
> `PathCollection` object holding all markers, which is why it stays fast even
> with hundreds of thousands of points. The `c` array flows through a
> normalization step into the colormap, mapping numbers to colors consistently.

**Syntax:**
```python
plt.scatter(x, y, s=sizes, c=color_values, cmap="viridis", alpha=0.8)
plt.colorbar(label="what the colors mean")   # decode c for the reader
```

**Example:** do students who study more score more? Dot size and color add a
third variable — sleep hours.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

study_hours = np.array([2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 8, 9])
marks       = np.array([45, 52, 55, 58, 62, 65, 70, 72, 75, 80, 86, 92])
sleep_hours = np.array([5, 6, 7, 6, 8, 7, 8, 9, 7, 8, 9, 8])   # third variable

sizes = (sleep_hours - 4) ** 2 * 12      # more sleep -> visibly bigger dot

plt.scatter(study_hours, marks, s=sizes, c=sleep_hours,
            cmap="viridis", alpha=0.8, edgecolor="black")
plt.colorbar(label="Sleep (hours/night)")

plt.title("Relation Question -> Scatter Plot")
plt.xlabel("Hours studied per week")
plt.ylabel("Exam marks")
plt.grid(True, linestyle="--", alpha=0.4)

plt.show()

# Dots drift upward to the right = positive relationship. Clear at a glance.

## 4. Grouped Bars — Comparing Categories Side by Side

Bars compare a *value across categories*. To put TWO series side by side, shift
each series off its integer slot by half a bar width, then re-label the axis.

**Syntax:**
```python
x = np.arange(len(categories))          # slots: 0, 1, 2, ...
w = 0.35                                # bar width
plt.bar(x - w/2, series_a, w, label="A")
plt.bar(x + w/2, series_b, w, label="B")
plt.xticks(x, categories)               # put names back under the slots
```

**Example:** monthly sales for two branches of a bookstore chain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

months     = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
dhaka      = [320, 290, 410, 380, 450, 520]
chattogram = [280, 310, 350, 420, 400, 480]

x = np.arange(len(months))   # 0..5 - one slot per month
w = 0.35                     # each bar takes 35% of a slot

plt.bar(x - w/2, dhaka,      w, label="Dhaka branch",      color="#2a6f97")
plt.bar(x + w/2, chattogram, w, label="Chattogram branch", color="#f4a261")

plt.xticks(x, months)                  # restore month names under slots
plt.title("Comparison Question -> Grouped Bars")
plt.xlabel("Month")
plt.ylabel("Books sold")
plt.legend()
plt.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.show()

## 5. Stacked Bars and Horizontal Bars

**Stacked bars** show parts contributing to a total: pass the running total via
`bottom=` so segments sit on top of each other instead of hiding behind.

**Horizontal bars** (`barh`) shine when category names are long — they get real
room on the y-axis instead of colliding.

**Syntax:**
```python
plt.bar(cats, series_a, label="A")
plt.bar(cats, series_b, bottom=series_a, label="B")   # stack B on top of A
plt.barh(cats, values)                                # sideways bars
```

**Example:** online vs retail share of those same book sales, then course
enrollments drawn horizontally.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
online = [180, 210, 260, 300, 340, 390]
retail = [420, 390, 500, 500, 510, 610]

x = np.arange(len(months))

plt.bar(x, online, label="Online orders", color="#2a9d8f")
plt.bar(x, retail, bottom=online, label="In-store sales", color="#e76f51")

plt.xticks(x, months)
plt.title("Stacked Sales Channels (total bar = full sales)")
plt.xlabel("Month")
plt.ylabel("Books sold")
plt.legend()
plt.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

courses  = ["Data Science with Python", "Machine Learning Basics",
            "Deep Learning", "Computer Vision"]
students = [240, 195, 150, 90]

plt.barh(courses, students, color="#6a4c93")

plt.title("Course Enrollments - long names fit easily")
plt.xlabel("Students enrolled")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.show()

## 6. Histograms vs Bar Charts — Don't Mix Them Up!

They look similar but answer different questions:

| | **Bar chart** | **Histogram** |
|---|---|---|
| X-axis holds | Categories (words) | Numeric ranges ("bins") |
| Bar height counts | One value per category | How many observations FALL IN each bin |
| Gaps between bars | Yes (they're separate things) | None — bins touch (the scale is continuous) |
| Example | Sales per branch | Distribution of exam scores |

A histogram slices a number line into intervals and counts what lands in each.
`bins=` controls the slice count: too few hides shape, too many shows noise.

**Syntax:**
```python
plt.hist(values, bins=10, edgecolor="white")             # count view
plt.hist(values, bins=10, density=True, alpha=0.6)       # probability view
```

What does `density=True` mean? Each bar's height becomes a *probability density*
so that the **total area of all bars equals exactly 1**. Counts depend on how
many samples you collected; densities don't — so this is the fair way to
overlay distributions from groups of different sizes.

**Example:** exam scores of one class of 40 students.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)                                   # deterministic scores
scores = np.clip(np.random.normal(72, 11, 40), 30, 100).round()

plt.hist(scores, bins=10, edgecolor="white", color="#457b9d")

plt.title("Distribution Question -> Histogram (class of 40)")
plt.xlabel("Score range")
plt.ylabel("Number of students")
plt.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.show()

# Most students cluster around 70; a thin tail struggles below 50.

In [ ]:
# Same scores, three bin counts - the story changes with your choice!
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
scores = np.clip(np.random.normal(72, 11, 40), 30, 100).round()

for n_bins in (3, 10, 25):
    plt.figure(figsize=(10, 3))
    plt.hist(scores, bins=n_bins, edgecolor="white", color="#457b9d")
    plt.title(f"bins={n_bins}  ({'too coarse' if n_bins == 3 else 'just right' if n_bins == 10 else 'too noisy'})")
    plt.xlabel("Score")
    plt.ylabel("Students")

plt.show()

# Rule: start near sqrt(number_of_points), then adjust until the shape is honest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
class_a = np.clip(np.random.normal(72, 11, 40), 30, 100).round()   # 40 students
class_b = np.clip(np.random.normal(64, 14, 18), 30, 100).round()   # only 18!

plt.hist(class_a, bins=10, density=True, alpha=0.6,
         edgecolor="white", label=f"Class A (n={len(class_a)})")
plt.hist(class_b, bins=10, density=True, alpha=0.6,
         edgecolor="white", label=f"Class B (n={len(class_b)})")

plt.title("Fair Comparison with density=True")
plt.xlabel("Score")
plt.ylabel("Density (each histogram's area sums to 1)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)

plt.show()

# With raw counts Class A would tower just because it has MORE STUDENTS.
# Density removes that unfair head start - only SHAPE remains comparable.

## 7. Pie Charts — Parts of One Whole (Handle with Care)

Pie charts show how a single total splits into shares. Key arguments:

| Argument | Job |
|---|---|
| `labels=` | Name each slice |
| `autopct="%1.1f%%"` | Print percentages on slices (`%1.1f` = one decimal, `%%` = literal % sign) |
| `explode=` | Pull chosen slice(s) outward, e.g. `[0.08, 0, 0, 0]` |
| `startangle=90` | Rotate so the first slice starts at 12 o'clock (convention) |
| `shadow=True` | ⚠️ Adds fake 3-D depth — looks dated and muddies edges. Keep it off. |

⚠️ Honest caveats: human eyes compare *angles* poorly. Keep pies to ≤ 5 slices;
if two slices are near-equal, a bar chart decides the winner far better.

**Syntax:**
```python
plt.pie(values, labels=names, autopct="%1.1f%%",
        explode=[0.08, 0, 0, 0], startangle=90)
```

**Example:** where a student's typical day actually goes.

In [ ]:
import matplotlib.pyplot as plt

activities = ["Classes", "Study", "Sleep", "Phone", "Other"]
minutes    = [300, 240, 450, 210, 240]

explode = [0, 0, 0, 0.08, 0]           # gently pull out the Phone slice

plt.pie(minutes,
        labels=activities,
        autopct="%1.1f%%",             # percentage on every slice
        explode=explode,
        startangle=90)                 # start at 12 o'clock

plt.title("A Student Day - 24 Hours Split")

plt.show()

In [ ]:
# The rematch: SAME data as a sorted horizontal bar chart.
# Which makes ranking easier - angles, or bar lengths?
import matplotlib.pyplot as plt

activities = ["Sleep", "Classes", "Study", "Other", "Phone"]
minutes    = [450, 300, 240, 240, 210]          # already sorted, largest first

plt.barh(activities[::-1], minutes[::-1], color="#457b9d")

plt.title("Same Data, Sorted Bars - instant ranking")
plt.xlabel("Minutes per day")

plt.show()

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Using `hist` for categorical data | Nonsense bins like "Dhaka=0..2" | `bar` for categories, `hist` for numbers |
| Grouped bars without `plt.xticks(x, months)` | Axis shows 0…5 instead of month names | Re-map labels after shifting positions |
| Stacked bar without `bottom=` | Second series paints OVER the first | Pass `bottom=first_series` |
| Pie with 8+ slices | Unreadable slivers and legend soup | Group small ones into "Other", or switch to a bar |
| Comparing two histograms with raw counts | Bigger group always looks "higher" | Use `density=True` for different-sized groups |
| Trusting default bin count blindly | Default bins may hide or invent structure | Choose `bins=` deliberately; try a few values |

## 💡 Best Practices & Pro Tips

- Sort bar categories by value (largest first) unless order carries meaning —
  sorted bars let readers rank instantly.
- Give histograms `edgecolor="white"` so adjacent bins stay visually separate.
- Always label your colorbar (`plt.colorbar(label=...)`); colored dots mean
  nothing without a key.
- One accent color beats seven colors: gray the context series, color the story.
- 🤖 **AI-engineering relevance:** before modeling you WILL make these exact four
  charts — histograms to spot skew/outliers (drives scaling choices), scatters to
  find correlated features (drives feature selection), bars for class balance
  (imbalanced datasets!), and loss curves over epochs.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `plt.scatter(x, y)` | One dot per observation — relations | `scatter(hours, marks)` |
| `s=`, `c=`, `cmap=`, `alpha=` | Size/color/blend per dot | `c=vals, cmap="viridis"` |
| `plt.colorbar(label=...)` | Decodes the `c` colors | `colorbar(label="Sleep")` |
| `plt.bar(x - w/2, ...)` / `+w/2` | Grouped side-by-side bars | see §4 |
| `bottom=` | Stacks bars on previous totals | `bar(x, b, bottom=a)` |
| `plt.barh(cats, vals)` | Horizontal bars for long names | see §5 |
| `plt.hist(vals, bins=10)` | Counts per numeric bin | `hist(scores, bins=10)` |
| `density=True` | Bars become probabilities (area = 1) | overlaying unequal groups |
| `plt.pie(vals, autopct="%1.1f%%", explode=..., startangle=90)` | Parts of a whole | see §7 |

**Key takeaways**

- Pick the chart from the QUESTION: trend/comparison/relation/distribution/parts-of-whole.
- Scatter encodes 4 dimensions: x, y, size (`s` = area!), color (`c`).
- Histogram ≠ bar chart: continuous bins vs distinct categories.
- Bin count is an editorial choice — coarse hides shape, fine invents noise.
- Pies are fine for ≤ 5 clean shares — and sorted bars usually say it better.

## 🔗 Next Lesson

➡️ Continue to **[04_Subplots](../04_Subplots/notes.ipynb)** — putting several charts in one figure, and the object-oriented style professionals use.